# selenium으로 동적웹페이지 데이터 수집하기_googleplay_금융앱 5개 리뷰수집

In [3]:
# !pip install selenium webdriver-manager

# 토스 앱 리뷰 수집하기_최종

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from datetime import datetime, timedelta
import pandas as pd
import time
from dbio import to_db

# selenium으로 웹브라우저 켜고 접속하는 함수

In [3]:
def create_driver(url):
    # Chrome 옵션 설정
    options = Options()
    options.add_argument("--window-size=1280,900")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    # ChromeDriver 설정
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(30)

    try:
        driver.get(f"https://play.google.com/store/apps/details?id={url}")
        print(f"{url} 접속 성공", end="\r")
    except Exception as e:
        print(e)
        print("driver 생성 실패")
            
    return driver

# 리뷰창 열고 최신순으로 정렬 후 지정한 날짜까지 스크롤하기

In [20]:
def get_reviews(driver, days):
    wait = WebDriverWait(driver, 15)

    # 자바스트립트로 윈도우를 0-1200px까지 스크롤 내리기
    driver.execute_script("window.scrollTo(0, 1200)")
    # 평점 및 리뷰 옆의 -> 버튼 찾아서 클릭하기
    xpath_selector = (
        '//button[contains(@aria-label, "평점 및 리뷰")] | '
        '//span[contains(text(), "평점 및 리뷰")]/ancestor::button | '
        '//header//i[contains(@class, "google-material-icons")]/parent::button'
    )

    # 기존의 특정 클래스명 대신, 리뷰 대화상자 전체를 찾아 스크롤합니다.
    try:
        # 1. 리뷰 목록이 담긴 컨테이너 찾기 (보통 role="dialog" 또는 특정 태그임)
        # 아래 스크립트는 화면에서 스크롤이 가능한 요소를 찾아 자동으로 내려줍니다.
        # 기존의 특정 클래스(.fysCi.Vk3ZVd) 대신 더 넓은 범위를 찾는 스크립트입니다.
    from selenium.webdriver.common.action_chains import ActionChains

    # 1. 현재 로드된 리뷰들 중 가장 마지막 요소를 찾습니다.
    reviews = driver.find_elements(By.CSS_SELECTOR, 'div[jscontroller="H6eYGe"]')

    if reviews:
        # 2. 마지막 리뷰 위치로 마우스를 옮기면 자동으로 스크롤 효과가 발생합니다.
        last_review = reviews[-1]
        actions = ActionChains(driver)
        actions.move_to_element(last_review).perform()
        time.sleep(2) # 추가 리뷰 로딩 대기
    except Exception as e:
        print(f"스크롤 중 오류 발생: {e}")

    while True:

        # 리뷰창 스크롤 내려서 과거 리뷰 로딩하기
        driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 1000)")
        time.sleep(1)
        # 리뷰 가장 마지막의 날짜 추출하기
        last_date = driver.find_elements(By.CSS_SELECTOR, ".bp9Aid")[-1].text
        last_date = last_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        last_date = datetime.strptime(last_date, "%Y-%m-%d")
        n_reviews = len(driver.find_elements(By.CSS_SELECTOR, ".bp9Aid"))
        print("리뷰개수:", n_reviews, "end_date", end_date, "last_date", last_date, end="\r")
        if last_date < end_date:
            break
    return driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")

IndentationError: expected an indented block after 'try' statement on line 14 (3429856753.py, line 18)

In [5]:
# 데이터 수집 및 DB 저장 함수
def review_extractnsave(items, url):
    all_reviews = []    
    # 스크롤이 멈춘 후 데이터 수집하기
    for item in driver.find_elements(By.CSS_SELECTOR, ".RHo1pe"):
        result = {} 
        # 리뷰에서 날짜 추출하기
        date = item.find_element(By.CSS_SELECTOR, ".bp9Aid").text
        date = date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        review_date = datetime.strptime(date, "%Y-%m-%d")
        # 평점 추출하기
        rating = item.find_element(By.CSS_SELECTOR, ".iXRFPc").get_attribute("aria-label")
        rating = rating.split()[3][0]
        # 리뷰 글 추출하기
        review_text = item.find_element(By.CSS_SELECTOR, ".h3YV2d").text

        result["date"] = review_date
        result['rating'] = rating
        result['review_text'] = review_text
        all_reviews.append(result)

    df = pd.DataFrame(all_reviews)
    display(df) 
    to_db("bank_app_reviews", f"{url}_reviews", df)

In [17]:
from selenium.webdriver.common.action_chains import ActionChains

# 1. 현재 로드된 리뷰들 중 가장 마지막 요소를 찾습니다.
reviews = driver.find_elements(By.CSS_SELECTOR, 'div[jscontroller="H6eYGe"]')

if reviews:
    # 2. 마지막 리뷰 위치로 마우스를 옮기면 자동으로 스크롤 효과가 발생합니다.
    last_review = reviews[-1]
    actions = ActionChains(driver)
    actions.move_to_element(last_review).perform()
    time.sleep(2) # 추가 리뷰 로딩 대기

In [21]:
urls = ["viva.republica.toss", "com.kebhana.hanapush", "com.kbstar.kbbank", "com.shinhan.sbanking", "com.wooribank.smart.npib"]
for url in urls:
    driver = create_driver(url)
    items = get_reviews(driver, 10)
    review_extractnsave(items, url)

JavascriptException: Message: javascript error: Cannot read properties of null (reading 'scrollBy')
  (Session info: chrome=145.0.7632.159)
Stacktrace:
#0 0x60315ecabb6a <unknown>
#1 0x60315e6bea32 <unknown>
#2 0x60315e6c5b62 <unknown>
#3 0x60315e6c86d9 <unknown>
#4 0x60315e75d409 <unknown>
#5 0x60315e75c369 <unknown>
#6 0x60315e705c0f <unknown>
#7 0x60315e7069d1 <unknown>
#8 0x60315ec706b9 <unknown>
#9 0x60315ec735c1 <unknown>
#10 0x60315ec5ce29 <unknown>
#11 0x60315ec7417e <unknown>
#12 0x60315ec434b0 <unknown>
#13 0x60315ec98578 <unknown>
#14 0x60315ec9874b <unknown>
#15 0x60315ecaa1a3 <unknown>
#16 0x7f0318e9caa4 <unknown>
#17 0x7f0318f29c6c <unknown>
